In [ ]:
!pip install requests pandas chromadb sentence-transformers httpx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 138.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 128.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/

In [ ]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

In [ ]:
def reconstruct_abstract(inv_idx):
    if not inv_idx:
        return ""

    words = {}

    for word, positions in inv_idx.items():
        for pos in positions:
            words[pos] = word

    return " ".join(words[i] for i in sorted(words))


def fetch_openalex(query, limit=10):
    url = "https://api.openalex.org/works"

    params = {
        "search": query,
        "per_page": limit
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()["results"]

    results = []

    for paper in data:
        results.append({
            "id": paper.get("id"),
            "title": paper.get("display_name"),
            "abstract": reconstruct_abstract(
                paper.get("abstract_inverted_index")
            ),
            "year": paper.get("publication_year"),
            "source": "openalex",
            "url": (
                paper.get("primary_location", {})
                .get("landing_page_url")
            )
        })

    return results

In [ ]:
openalex_papers = fetch_openalex("smart agriculture")
openalex_papers[:2]

[{'id': 'https://openalex.org/W2078771477',
  'title': 'Climate-smart agriculture for food security',
  'abstract': '',
  'year': 2014,
  'source': 'openalex',
  'url': 'https://doi.org/10.1038/nclimate2437'},
 {'id': 'https://openalex.org/W2964535757',
  'title': 'Internet-of-Things (IoT)-Based Smart Agriculture: Toward Making the Fields Talk',
  'abstract': "Despite the perception people may have regarding the agricultural process, the reality is that today's agriculture industry is data-centered, precise, and smarter than ever. The rapid emergence of the Internet-of-Things (IoT) based technologies redesigned almost every industry including “smart agriculture” which moved the industry from statistical to quantitative approaches. Such revolutionary changes are shaking the existing agriculture methods and creating new opportunities along a range of challenges. This article highlights the potential of wireless sensors and IoT in agriculture, as well as the challenges expected to be face

In [ ]:
def fetch_arxiv(query, limit=10):
    url = "http://export.arxiv.org/api/query"

    params = {
        "search_query": f"all:{query}",
        "start": 0,
        "max_results": limit
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    root = ET.fromstring(response.text)

    ns = {"atom": "http://www.w3.org/2005/Atom"}

    results = []

    for entry in root.findall("atom:entry", ns):

        results.append({
            "id": entry.find("atom:id", ns).text,
            "title": entry.find("atom:title", ns).text.strip(),
            "abstract": entry.find("atom:summary", ns).text.strip(),
            "year": int(
                entry.find("atom:published", ns).text[:4]
            ),
            "source": "arxiv",
            "url": entry.find("atom:id", ns).text
        })

    return results

In [ ]:
arxiv_papers = fetch_arxiv("smart agriculture")
arxiv_papers[:2]

[{'id': 'http://arxiv.org/abs/2506.10106v1',
  'title': 'One For All: LLM-based Heterogeneous Mission Planning in Precision Agriculture',
  'abstract': 'Artificial intelligence is transforming precision agriculture, offering farmers new tools to streamline their daily operations. While these technological advances promise increased efficiency, they often introduce additional complexity and steep learning curves that are particularly challenging for non-technical users who must balance tech adoption with existing workloads. In this paper, we present a natural language (NL) robotic mission planner that enables non-specialists to control heterogeneous robots through a common interface. By leveraging large language models (LLMs) and predefined primitives, our architecture seamlessly translates human language into intermediate descriptions that can be executed by different robotic platforms. With this system, users can formulate complex agricultural missions without writing any code. In the

In [ ]:
import requests

def fetch_patentsview(query, limit=10):
    url = "https://api.patentsview.org/patents/query"

    payload = {
        "q": {
            "_text_any": {
                "patent_title": query
            }
        },
        "f": [
            "patent_id",
            "patent_title",
            "patent_abstract",
            "patent_date"
        ],
        "o": {
            "per_page": limit
        }
    }

    response = requests.post(url, json=payload)

    print("Status Code:", response.status_code)
    print("Response Preview:")
    print(response.text[:1000])

    return response

In [ ]:
fetch_patentsview("smart agriculture")

Status Code: 200
Response Preview:
<!doctype html>
<html lang="en" data-beasties-container>
<head><link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
  <meta charset="utf-8">
  <base href="/">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <meta name="description" content="The Open Data Portal (ODP) is USPTO's data platform that empowers you to discover and easily extract USPTO data in one place for free.">

  <!-- adds cache control -->
  <meta http-equiv="Cache-Control" content="no-cache, no-store, must-revalidate">
  <meta http-equiv="pragma" content="no-cache">
  <meta http-equiv="expires" content="-1">

  <link rel="icon" href="/favicon.ico" sizes="any">
  <link rel="shortcut icon" type="image/x-icon" href="assets/images/icons/favicon.ico">

  <script src="https://code.jquery.com/jquery-3.7.1.slim.min.js"></script>
  <!-- <script src="https://cdn.jsdelivr.net/npm/bootstrap@4.5.3/dist/js/bootstrap.bundle.min.js"></script> -->

  <script>
 

<Response [200]>

In [ ]:
query = "smart agriculture"

all_documents = openalex_papers + arxiv_papers

print("OpenAlex:", len(openalex_papers))
print("arXiv:", len(arxiv_papers))
print("Total:", len(all_documents))

OpenAlex: 10
arXiv: 10
Total: 20


In [ ]:
df = pd.DataFrame(all_documents)

print(df.shape)

df.head()

(20, 6)


,id,title,abstract,year,source,url
0,https://openalex.org/W2078771477,Climate-smart agriculture for food security,,2014,openalex,https://doi.org/10.1038/nclimate2437
1,https://openalex.org/W2964535757,Internet-of-Things (IoT)-Based Smart Agricultu...,Despite the perception people may have regardi...,2019,openalex,https://doi.org/10.1109/access.2019.2932609
2,https://openalex.org/W2291715071,Climate-smart agriculture: sourcebook.,,2013,openalex,https://www.cabdirect.org/abstracts/2015323730...
3,https://openalex.org/W3044921708,Unmanned Aerial Vehicles in Smart Agriculture:...,"In the next few years, smart farming will reac...",2021,openalex,https://doi.org/10.1109/jsen.2021.3049471
4,https://openalex.org/W3138616181,Internet of Things for the Future of Smart Agr...,This paper presents a comprehensive review of ...,2021,openalex,https://doi.org/10.1109/jas.2021.1003925


In [ ]:
df.columns

Index(['id', 'title', 'abstract', 'year', 'source', 'url'], dtype='object')

In [ ]:
df["abstract"] = df["abstract"].fillna("").str.strip()

df = df[df["abstract"] != ""].reset_index(drop=True)

print("Remaining documents:", len(df))

Remaining documents: 17


In [ ]:
df["title_lower"] = df["title"].str.lower().str.strip()

df = df.drop_duplicates(
    subset=["title_lower"]
).reset_index(drop=True)

df.drop(columns=["title_lower"], inplace=True)

print("Unique documents:", len(df))

Unique documents: 17


In [ ]:
!pip install -q chromadb sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [ ]:
def chunk_text(text,
               chunk_size=500,
               overlap=100):

    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks

In [ ]:
print(chunk_text(df.loc[0, "abstract"]))

["Despite the perception people may have regarding the agricultural process, the reality is that today's agriculture industry is data-centered, precise, and smarter than ever. The rapid emergence of the Internet-of-Things (IoT) based technologies redesigned almost every industry including “smart agriculture” which moved the industry from statistical to quantitative approaches. Such revolutionary changes are shaking the existing agriculture methods and creating new opportunities along a range of ch", 'nges are shaking the existing agriculture methods and creating new opportunities along a range of challenges. This article highlights the potential of wireless sensors and IoT in agriculture, as well as the challenges expected to be faced when integrating this technology with the traditional farming practices. IoT devices and communication techniques associated with wireless sensors encountered in agriculture applications are analyzed in detail. What sensors are available for specific agri

In [ ]:
import chromadb

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="research_collection"
)

print("Collection created.")

Collection created.


In [ ]:
doc_counter = 0

for _, row in df.iterrows():

    chunks = chunk_text(row["abstract"])

    for chunk in chunks:

        embedding = embedding_model.encode(
            chunk
        ).tolist()

        collection.add(
            ids=[f"doc_{doc_counter}"],

            documents=[chunk],

            embeddings=[embedding],

            metadatas=[{
                "title": row["title"],
                "year": int(row["year"]),
                "source": row["source"],
                "domain": query,
                "url": row["url"]
            }]
        )

        doc_counter += 1

print("Stored chunks:", collection.count())

Stored chunks: 60


In [ ]:
def retrieve(query_text, top_k=5):

    query_embedding = embedding_model.encode(
        query_text
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    return results

In [ ]:
results = retrieve(
    "AI for crop disease prediction",
    top_k=3
)

In [ ]:
for i in range(len(results["documents"][0])):

    print("=" * 80)

    print("Result:", i + 1)

    print("\nTitle:")
    print(results["metadatas"][0][i]["title"])

    print("\nSource:")
    print(results["metadatas"][0][i]["source"])

    print("\nYear:")
    print(results["metadatas"][0][i]["year"])

    print("\nChunk:")
    print(results["documents"][0][i])

    print("\nURL:")
    print(results["metadatas"][0][i]["url"])

    print()

Result: 1

Title:
Internet-of-Things (IoT)-Based Smart Agriculture: Toward Making the Fields Talk

Source:
openalex

Year:
2019

Chunk:
 agriculture applications are analyzed in detail. What sensors are available for specific agriculture application, like soil preparation, crop status, irrigation, insect and pest detection are listed. How this technology helping the growers throughout the crop stages, from sowing until harvesting, packing and transportation is explained. Furthermore, the use of unmanned aerial vehicles for crop surveillance and other favorable applications such as optimizing crop yield is considered in this arti

URL:
https://doi.org/10.1109/access.2019.2932609

Result: 2

Title:
IOT Based Monitoring System in Smart Agriculture

Source:
openalex

Year:
2017

Chunk:
Internet of Things (IoT) plays a crucial role in smart agriculture. Smart farming is an emerging concept, because IoT sensors capable of providing information about their agriculture fields. The paper aims m